# Model 3: RAG (Retrieval-Augmented Generation)

**Goal:** For each question, first *find* the relevant Austrian tax law text, then give it to GPT as context.

**Analogy:** Model 1 = closed-book exam. Model 3 = open-book exam (GPT can look up the law).

**3 steps:**
1. Load Austrian tax law texts from local PDF files (KStG, EStG, UStG)
2. Embed the law paragraphs as vectors so we can search them
3. For each question: find the most relevant paragraphs → ask GPT with that context

In [ ]:
# Install required packages (run once)
!pip install openai pandas numpy pdfplumber

In [ ]:
import pandas as pd
import numpy as np
import re
import pdfplumber
from openai import OpenAI

# ⚠ Replace with your API key — do not commit this to GitHub!
API_KEY = "INSERT-YOUR-KEY-HERE"
client = OpenAI(api_key=API_KEY)

## Step 1: Load Austrian Tax Law Texts from Local PDFs

We read the 3 law texts directly from the PDF files saved in `Context/Gesetze/`.

In [ ]:
def read_pdf(path):
    """Extract all text from a PDF file."""
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    print(f"Read {path.split('/')[-1]}: {len(text)} characters")
    return text

kstg_text = read_pdf("../../Context/Gesetze/KStG 1988, Fassung vom 03.04.2026.pdf")
estg_text = read_pdf("../../Context/Gesetze/EStG.pdf")
ustg_text = read_pdf("../../Context/Gesetze/UStG.pdf")

In [ ]:
def split_into_chunks(text, min_length=100):
    """Split law text into paragraphs. Each paragraph starts with a § symbol."""
    chunks = re.split(r'(?=§\s*\d+)', text)
    chunks = [c.strip() for c in chunks if len(c.strip()) > min_length]
    return chunks

kstg_chunks = split_into_chunks(kstg_text)
estg_chunks = split_into_chunks(estg_text)
ustg_chunks = split_into_chunks(ustg_text)
all_chunks  = kstg_chunks + estg_chunks + ustg_chunks

print(f"KStG chunks: {len(kstg_chunks)}")
print(f"EStG chunks: {len(estg_chunks)}")
print(f"UStG chunks: {len(ustg_chunks)}")
print(f"Total chunks: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[0][:300])

## Step 2: Embed the Law Paragraphs

We convert each law paragraph into a vector (a list of numbers) using OpenAI's embedding model.
Later, we do the same for each question and find which paragraphs are closest = most relevant.

In [ ]:
def get_embedding(text):
    """Convert text to a vector using OpenAI's embedding model."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text[:2000]
    )
    return response.data[0].embedding

print(f"Embedding {len(all_chunks)} law paragraphs... (this takes a few minutes)")

chunk_embeddings = []
for i, chunk in enumerate(all_chunks):
    emb = get_embedding(chunk)
    chunk_embeddings.append(emb)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(all_chunks)} done")

chunk_embeddings = np.array(chunk_embeddings)
print(f"\nDone! Shape: {chunk_embeddings.shape}")

## Step 3: Run RAG Inference

For each question:
1. Embed the question as a vector
2. Find the 3 law paragraphs most similar to the question (cosine similarity)
3. Give those paragraphs to GPT as context, then ask GPT the question

In [ ]:
def find_relevant_chunks(question, k=3):
    """Find the k most relevant law paragraphs for a question."""
    q_emb = np.array(get_embedding(question))
    scores = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb) + 1e-9
    )
    top_k_idx = np.argsort(scores)[-k:][::-1]
    return [all_chunks[i] for i in top_k_idx]

# Test the retrieval
test_q = "Was ist die Bemessungsgrundlage für die Körperschaftsteuer?"
test_result = find_relevant_chunks(test_q)
print(f"Test question: {test_q}")
print(f"\nMost relevant paragraph (first 300 chars):")
print(test_result[0][:300])

In [ ]:
# Load the dataset
df = pd.read_csv("../../dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [ ]:
# Run RAG inference: retrieve relevant law text, then ask GPT
system_prompt = """Du bist ein österreichischer Steuerrechtsexperte.
Nutze die folgenden Gesetzestexte als Kontext und beantworte die Frage präzise.
Nenne die relevanten Paragraphen (z.B. § 7 Abs. 1 KStG).
Antworte auf Deutsch in 1–3 kurzen Sätzen."""

answers = []

for i, row in df.iterrows():
    context_chunks = find_relevant_chunks(row["prompt"], k=3)
    context = "\n\n".join(context_chunks)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Gesetzestext:\n{context}\n\nFrage: {row['prompt']}"}
        ],
        max_completion_tokens=300,
        temperature=0
    )
    answer = response.choices[0].message.content.strip()
    answers.append(answer)
    print(f"{i+1}/{len(df)} | {row['id']} | {answer[:70]}...")

In [ ]:
# Save results
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("../results/model3_results.csv", index=False)
print(f"Saved {len(results)} answers to model3_results.csv")
results.head(3)